# DenoiseRL × Nemotron Submission Notebook

**Purpose:** train a Kaggle-valid LoRA adapter using a DenoiseRL-style recovery curriculum:

1. Build a fixed pool `W(q)` of wrong / noisy reasoning prefixes.
2. Mix normal solve-from-scratch examples with denoise/recovery examples.
3. Mask the noisy off-policy prefix from loss.
4. Train only the policy continuation to recover and emit a verified `\boxed{...}` answer.
5. Save and validate a rank-32-or-lower PEFT adapter package.

This notebook is designed for the NVIDIA Nemotron Model Reasoning Challenge style workflow, where the final submission is a LoRA adapter, not a prediction CSV.

**Default mode:** resource-safe DenoiseSFT / DenoisePreference approximation for Kaggle adapter training.  
**Optional mode:** online DenoiseGRPO core is included and guarded because full RL rollouts on 30B-class models require heavy GPU memory.

In [ ]:
# ============================================================
# 0. Environment bootstrap
# ============================================================
# Run this cell on Kaggle only when packages are missing. It does not fabricate data.
# Internet may be disabled in scored notebooks; attach wheel datasets if needed.

import os, sys, json, math, re, time, random, shutil, zipfile, glob, gc, hashlib, inspect
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Iterable

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 918
random.seed(SEED)

try:
    import numpy as np
    np.random.seed(SEED)
except Exception:
    np = None

try:
    import pandas as pd
except Exception as e:
    raise RuntimeError("pandas is required. Attach/install pandas before running this notebook.") from e

try:
    import torch
except Exception as e:
    raise RuntimeError("PyTorch is required for training.") from e

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class CFG:
    # Competition/model paths
    input_root: str = "/kaggle/input"
    work_root: str = "/kaggle/working"
    output_dir: str = "/kaggle/working/denoiserl_nemotron_adapter"
    package_zip: str = "/kaggle/working/denoiserl_nemotron_adapter.zip"

    # Base model: can be overridden by environment or auto-discovery.
    base_model_name_or_path: str = os.environ.get("BASE_MODEL_PATH", "nvidia/Nemotron-3-Nano-30B-A3B-BF16")
    weak_model_name_or_path: str = os.environ.get("WEAK_MODEL_PATH", "")

    # Dataset discovery. Leave blank for auto-discovery.
    train_file: str = os.environ.get("TRAIN_FILE", "")
    valid_file: str = os.environ.get("VALID_FILE", "")
    public_trace_file: str = os.environ.get("PUBLIC_TRACE_FILE", "")

    # DenoiseRL knobs from paper, adapted to adapter training.
    main_rollouts_n: int = 12
    denoise_rollouts_k: int = 4
    prefix_ratio_rho: float = 0.20
    response_budget_tokens: int = 2048       # use 4096+ when GPU budget allows
    max_seq_length: int = 2048               # set 4096/8192 on A100/H100 if possible
    min_wrong_prefixes_per_row: int = 4
    max_wrong_prefixes_per_row: int = 8

    # Training scale guards.
    max_train_rows: int = int(os.environ.get("MAX_TRAIN_ROWS", "0"))  # 0 = all
    max_records: int = int(os.environ.get("MAX_RECORDS", "0"))        # 0 = all
    train_time_budget_hours: float = float(os.environ.get("TRAIN_TIME_BUDGET_HOURS", "3.80"))

    # PEFT/QLoRA.
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.03
    learning_rate: float = float(os.environ.get("LR", "1.0e-4"))
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    num_train_epochs: float = float(os.environ.get("EPOCHS", "1.0"))
    per_device_train_batch_size: int = int(os.environ.get("BS", "1"))
    gradient_accumulation_steps: int = int(os.environ.get("GA", "16"))
    save_steps: int = int(os.environ.get("SAVE_STEPS", "100"))
    logging_steps: int = 10

    # Runtime flags.
    use_4bit: bool = os.environ.get("USE_4BIT", "1") == "1"
    bf16: bool = os.environ.get("BF16", "1") == "1"
    gradient_checkpointing: bool = True
    train_adapter: bool = os.environ.get("TRAIN_ADAPTER", "1") == "1"
    run_local_generation_eval: bool = os.environ.get("RUN_EVAL", "0") == "1"
    run_online_grpo: bool = os.environ.get("RUN_ONLINE_GRPO", "0") == "1"

    # Safety and submission validation.
    require_real_train_data: bool = True
    enforce_rank_limit: bool = True
    max_lora_rank: int = 32

cfg = CFG()
Path(cfg.work_root).mkdir(parents=True, exist_ok=True)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))
assert cfg.lora_r <= cfg.max_lora_rank, f"LoRA rank {cfg.lora_r} exceeds allowed max {cfg.max_lora_rank}"
assert 0 < cfg.prefix_ratio_rho <= 1.0

In [ ]:
# ============================================================
# 2. Robust file and dataset discovery
# ============================================================

QUESTION_CANDIDATES = ["question", "problem", "prompt", "input", "query", "text", "task", "messages"]
ANSWER_CANDIDATES   = ["answer", "target", "label", "solution", "output", "gt", "ground_truth", "final_answer"]
ID_CANDIDATES       = ["id", "uid", "problem_id", "question_id", "sample_id"]
TRACE_CANDIDATES    = ["reasoning", "trace", "cot", "solution_trace", "rationale", "trajectory"]
WRONG_PREFIX_CANDIDATES = [
    "wrong_answer_with_boxed", "wrong_prefix", "wrong_prefixes", "wrong_trace", "wrong_traces",
    "incorrect_reasoning", "negative_trace", "bad_trace", "weak_rollouts", "weak_wrong_rollouts"
]

SUPPORTED_EXTS = {".csv", ".json", ".jsonl", ".parquet"}

def list_candidate_data_files(root: str) -> List[Path]:
    paths = []
    rootp = Path(root)
    if not rootp.exists():
        return []
    for p in rootp.rglob("*"):
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTS:
            # ignore hidden/system tiny files when possible
            if p.name.startswith("."):
                continue
            paths.append(p)
    return sorted(paths, key=lambda p: (p.stat().st_size, str(p)), reverse=True)

def read_any_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suf = path.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(path)
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf == ".jsonl":
        return pd.read_json(path, lines=True)
    if suf == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            # Common layouts: {data:[...]}, {train:[...]}, {id:{...}}
            for key in ["data", "train", "examples", "records", "rows"]:
                if key in data and isinstance(data[key], list):
                    return pd.DataFrame(data[key])
            if all(isinstance(v, dict) for v in data.values()):
                rows = []
                for k, v in data.items():
                    row = dict(v)
                    row.setdefault("id", k)
                    rows.append(row)
                return pd.DataFrame(rows)
            return pd.DataFrame([data])
    raise ValueError(f"Unsupported file: {path}")

def pick_col(cols: Iterable[str], candidates: List[str]) -> Optional[str]:
    lower = {str(c).lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    # fuzzy contains only after exact pass
    for c in cols:
        lc = str(c).lower()
        if any(cand in lc for cand in candidates):
            return c
    return None

def score_dataset_frame(df: pd.DataFrame) -> int:
    if df is None or df.empty:
        return -10**9
    score = 0
    cols = list(df.columns)
    if pick_col(cols, QUESTION_CANDIDATES): score += 10
    if pick_col(cols, ANSWER_CANDIDATES): score += 10
    if pick_col(cols, WRONG_PREFIX_CANDIDATES): score += 4
    if pick_col(cols, TRACE_CANDIDATES): score += 3
    score += min(len(df), 10000) // 100
    return score

def discover_train_file(cfg: CFG) -> Path:
    if cfg.train_file:
        p = Path(cfg.train_file)
        if not p.exists():
            raise FileNotFoundError(f"TRAIN_FILE does not exist: {p}")
        return p
    candidates = list_candidate_data_files(cfg.input_root)
    ranked = []
    for p in candidates:
        name = p.name.lower()
        if any(bad in name for bad in ["sample_submission", "submission"]):
            continue
        try:
            df = read_any_table(p)
            s = score_dataset_frame(df)
            # boost obvious train files
            if "train" in str(p).lower(): s += 8
            if "valid" in str(p).lower(): s += 2
            ranked.append((s, p, df.shape, list(df.columns)[:12]))
        except Exception as e:
            continue
    ranked.sort(reverse=True, key=lambda x: x[0])
    print("Top dataset candidates:")
    for s, p, shape, cols in ranked[:10]:
        print(f"  score={s:4d} shape={shape} path={p} cols={cols}")
    if not ranked or ranked[0][0] < 15:
        raise FileNotFoundError("Could not find a real train dataset with question+answer columns under /kaggle/input. Attach the competition train file or set TRAIN_FILE.")
    return ranked[0][1]

train_path = discover_train_file(cfg)
train_df_raw = read_any_table(train_path)
print("Selected train_path:", train_path)
print("Raw shape:", train_df_raw.shape)
print("Columns:", list(train_df_raw.columns))

In [ ]:
# ============================================================
# 3. Normalization, boxed extraction, verification
# ============================================================

def stringify_question(x: Any) -> str:
    if isinstance(x, list):
        # message list or list of fragments
        parts = []
        for item in x:
            if isinstance(item, dict):
                role = item.get("role", "")
                content = item.get("content", "")
                parts.append(f"{role}: {content}" if role else str(content))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    return "" if pd.isna(x) else str(x)

def normalize_answer(ans: Any) -> str:
    if ans is None:
        return ""
    s = str(ans).strip()
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("$", "").strip()
    # Keep tuples and fractions intact, remove harmless spaces/commas in large numerals.
    s = re.sub(r"\s+", "", s)
    if re.fullmatch(r"[-+]?\d{1,3}(,\d{3})+(\.\d+)?", s):
        s = s.replace(",", "")
    return s

def extract_boxed(text: Any) -> str:
    s = "" if text is None else str(text)
    # Balanced-ish boxed parser.
    marker = "\\boxed"
    idx = s.rfind(marker)
    if idx == -1:
        # heuristic: final answer tag
        m = re.findall(r"(?i)(?:final\s*answer|answer)\s*[:=]\s*([^\n]+)", s)
        if m:
            return normalize_answer(m[-1])
        nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:/\d+)?", s)
        return normalize_answer(nums[-1]) if nums else ""
    brace = s.find("{", idx)
    if brace == -1:
        return ""
    depth = 0
    out = []
    for ch in s[brace:]:
        if ch == "{":
            depth += 1
            if depth > 1: out.append(ch)
        elif ch == "}":
            depth -= 1
            if depth == 0: break
            out.append(ch)
        else:
            if depth >= 1: out.append(ch)
    return normalize_answer("".join(out))

def maybe_float(s: str) -> Optional[float]:
    s = normalize_answer(s)
    if not s:
        return None
    try:
        if "/" in s and re.fullmatch(r"[-+]?\d+/[-+]?\d+", s):
            a, b = s.split("/")
            return float(a) / float(b)
        return float(s)
    except Exception:
        return None

def answers_match(pred: Any, gold: Any, rel_tol: float = 1e-4, abs_tol: float = 1e-4) -> bool:
    p = normalize_answer(extract_boxed(pred) if "boxed" in str(pred) else pred)
    g = normalize_answer(gold)
    if p == g:
        return True
    pf, gf = maybe_float(p), maybe_float(g)
    if pf is not None and gf is not None:
        return math.isclose(pf, gf, rel_tol=rel_tol, abs_tol=abs_tol)
    return p.lower() == g.lower()

def boxed(ans: Any) -> str:
    return "\\boxed{" + str(ans).strip() + "}"

# Unit tests for critical parser/verifier behavior.
assert extract_boxed("final is \\boxed{123}") == "123"
assert extract_boxed("x \\boxed{\\frac{7}{15}}") == "\\frac{7}{15}"
assert answers_match("\\boxed{1.00001}", "1", rel_tol=1e-3)
print("Parser/verifier tests passed.")

In [ ]:
# ============================================================
# 4. Canonicalize training rows
# ============================================================

def canonicalize_training_df(df: pd.DataFrame, source_path: Path) -> pd.DataFrame:
    cols = list(df.columns)
    qcol = pick_col(cols, QUESTION_CANDIDATES)
    acol = pick_col(cols, ANSWER_CANDIDATES)
    idcol = pick_col(cols, ID_CANDIDATES)
    tracecol = pick_col(cols, TRACE_CANDIDATES)
    wrongcol = pick_col(cols, WRONG_PREFIX_CANDIDATES)
    if not qcol or not acol:
        raise ValueError(f"Dataset must contain question+answer columns. Found q={qcol}, a={acol}, cols={cols}")
    out = pd.DataFrame()
    out["id"] = df[idcol].astype(str) if idcol else [f"row_{i}" for i in range(len(df))]
    out["question"] = df[qcol].map(stringify_question)
    out["answer"] = df[acol].map(normalize_answer)
    out["source_path"] = str(source_path)
    out["gold_trace"] = df[tracecol].map(lambda x: "" if pd.isna(x) else str(x)) if tracecol else ""
    out["external_wrong_prefixes"] = df[wrongcol] if wrongcol else None
    out = out[(out.question.str.len() > 0) & (out.answer.str.len() > 0)].copy()
    out.drop_duplicates(subset=["question", "answer"], inplace=True)
    if cfg.max_train_rows and len(out) > cfg.max_train_rows:
        out = out.sample(cfg.max_train_rows, random_state=SEED).reset_index(drop=True)
    else:
        out = out.reset_index(drop=True)
    if cfg.require_real_train_data and len(out) == 0:
        raise RuntimeError("No real training rows survived canonicalization.")
    return out

train_df = canonicalize_training_df(train_df_raw, train_path)
print("Canonical train rows:", len(train_df))
print(train_df.head(3)[["id", "question", "answer"]].to_string(max_colwidth=160))

In [ ]:
# ============================================================
# 5. Category routing: CPCR + DenoiseRL difficulty control
# ============================================================

CATEGORY_ORDER = [
    "numeral_system", "unit_conversion", "physics_gravity", "equation_numeric",
    "bit_manipulation", "cipher", "cryptarithm", "logic", "general"
]

CATEGORY_WEIGHTS = {
    # Strong deterministic categories need retention but fewer denoise perturbations.
    "numeral_system": 0.85,
    "unit_conversion": 0.90,
    "physics_gravity": 0.95,
    "equation_numeric": 1.25,
    # Plateau categories receive extra recovery examples.
    "bit_manipulation": 1.40,
    "cipher": 1.50,
    "cryptarithm": 1.65,
    "logic": 1.15,
    "general": 1.00,
}

def classify_category(question: str) -> str:
    q = question.lower()
    if any(k in q for k in ["cipher", "encrypted", "decrypt", "substitution", "encoded", "code word", "caesar"]):
        return "cipher"
    if any(k in q for k in ["cryptarithm", "different digit", "letters represent", "send + more", "alphametic"]):
        return "cryptarithm"
    if any(k in q for k in ["bit", "xor", "and", "or", "shift", "binary", "mask", "rotate"]):
        return "bit_manipulation"
    if any(k in q for k in ["base ", "base-", "binary", "octal", "hexadecimal", "decimal representation", "numeral"]):
        return "numeral_system"
    if any(k in q for k in ["convert", "meters", "feet", "inches", "kilogram", "pounds", "liters", "gallons", "celsius", "fahrenheit"]):
        return "unit_conversion"
    if any(k in q for k in ["gravity", "gravitational", "planet", "mass", "radius", "newton", "orbit"]):
        return "physics_gravity"
    if any(k in q for k in ["solve", "equation", "polynomial", "system of", "x =", "y =", "compute", "evaluate"]):
        return "equation_numeric"
    if any(k in q for k in ["always true", "therefore", "logical", "knight", "liar"]):
        return "logic"
    return "general"

train_df["category"] = train_df.question.map(classify_category)
print(train_df.category.value_counts().to_string())

In [ ]:
# ============================================================
# 6. Wrong-prefix pool W(q)
# ============================================================
# Sources, in priority order:
#   1) External weak-model wrong rollouts already attached in the dataset.
#   2) Optional weak model generation, if RUN_WEAK_COLLECTION=1 and a weak model exists.
#   3) Deterministic verifier-aware perturbations of verified answers/traces.
#      These are not mock data: they are structured corruptions of real train rows.

_DIGITS = "0123456789"

def parse_external_wrong_prefixes(value: Any) -> List[str]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    if isinstance(value, dict):
        vals = []
        for v in value.values():
            if isinstance(v, list): vals.extend(v)
            else: vals.append(v)
        return [str(x) for x in vals if str(x).strip()]
    s = str(value).strip()
    if not s:
        return []
    if s.startswith("[") or s.startswith("{"):
        try:
            return parse_external_wrong_prefixes(json.loads(s))
        except Exception:
            pass
    # split multi-trace payloads conservatively
    parts = re.split(r"\n\s*---+\s*\n|\n\s*###\s*\n", s)
    return [p.strip() for p in parts if p.strip()]

def numeric_neighbors(ans: str) -> List[str]:
    s = normalize_answer(ans)
    out = []
    f = maybe_float(s)
    if f is not None:
        if float(f).is_integer():
            n = int(round(f))
            out += [str(n + 1), str(n - 1), str(-n), str(n * 10), str(n // 10 if n else 1)]
            if abs(n) > 1: out += [str(n % 10), str(n + 9)]
        else:
            out += [str(f + 1), str(f * 10), str(round(f, 1))]
    digits = re.findall(r"\d", s)
    if len(digits) >= 2:
        swapped = list(s)
        idxs = [i for i, ch in enumerate(s) if ch.isdigit()]
        swapped[idxs[0]], swapped[idxs[-1]] = swapped[idxs[-1]], swapped[idxs[0]]
        out.append("".join(swapped))
    return list(dict.fromkeys([x for x in out if normalize_answer(x) != s and x.strip()]))

def string_neighbors(ans: str) -> List[str]:
    s = str(ans).strip()
    out = []
    if not s: return out
    if len(s) > 1:
        out += [s[:-1], s[1:], s[::-1]]
    out += [s + "0", "0" + s]
    trans = str.maketrans("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz", "BCDEFGHIJKLMNOPQRSTUVWXYZAabcdefghijklmnopqrstuvwxyza")
    out.append(s.translate(trans))
    return list(dict.fromkeys([x for x in out if normalize_answer(x) != normalize_answer(s) and x.strip()]))

def category_wrong_answers(question: str, answer: str, category: str) -> List[str]:
    outs = []
    outs.extend(numeric_neighbors(answer))
    if category == "bit_manipulation":
        vals = re.findall(r"\b\d+\b", question)
        nums = [int(v) for v in vals[:8]]
        if nums:
            acc = nums[0]
            for n in nums[1:]: acc ^= n
            outs.append(str(acc))
            acc = nums[0]
            for n in nums[1:]: acc += n
            outs.append(str(acc))
            outs.extend([str(nums[0] << 1), str(nums[0] >> 1)])
    elif category == "unit_conversion":
        f = maybe_float(answer)
        if f is not None:
            outs.extend([str(f * 2.54), str(f / 2.54), str(f * 1000), str(f / 1000)])
    elif category == "cipher":
        outs.extend(string_neighbors(answer))
    elif category == "cryptarithm":
        outs.extend(string_neighbors(answer))
        f = maybe_float(answer)
        if f is not None:
            outs.extend([str(int(f) + 11), str(int(f) - 11)])
    elif category == "numeral_system":
        f = maybe_float(answer)
        if f is not None and float(f).is_integer():
            n = int(f)
            outs.extend([bin(n)[2:], hex(n)[2:].upper(), oct(n)[2:]])
    outs.extend(string_neighbors(answer))
    # Dedup and remove correct answer.
    clean = []
    for x in outs:
        x = normalize_answer(x)
        if x and not answers_match(x, answer) and x not in clean:
            clean.append(x)
    return clean[:16]

def build_wrong_prefix(question: str, answer: str, wrong_answer: str, category: str, style_id: int) -> str:
    # Prefix intentionally ends in a wrong intermediate state; it is visible but masked during training.
    templates = [
        "I will solve this quickly. I follow the first apparent pattern and get {wrong}. Therefore the answer seems to be {box}.",
        "Start from the obvious relation. After simplifying, I obtain {wrong}. I will proceed with this value: {box}.",
        "A direct calculation gives {wrong}. This appears consistent, so the provisional final answer is {box}.",
        "Using the most tempting shortcut for this {category} problem, I conclude {box}. Continuing from here,",
        "The intermediate state appears to force the value {wrong}, so I write {box} and move on.",
    ]
    t = templates[style_id % len(templates)]
    return t.format(wrong=wrong_answer, box=boxed(wrong_answer), category=category)

def truncate_prefix_by_ratio(prefix: str, rho: float, tokenizer=None, min_tokens: int = 8, max_tokens: Optional[int] = None) -> str:
    prefix = str(prefix).strip()
    if not prefix:
        return prefix
    if tokenizer is not None:
        ids = tokenizer(prefix, add_special_tokens=False).input_ids
        p = max(1, int(math.floor(rho * len(ids))))
        p = max(min_tokens, p) if len(ids) >= min_tokens else len(ids)
        if max_tokens: p = min(p, max_tokens)
        return tokenizer.decode(ids[:p], skip_special_tokens=True).strip()
    words = prefix.split()
    p = max(1, int(math.floor(rho * len(words))))
    p = max(min_tokens, p) if len(words) >= min_tokens else len(words)
    if max_tokens: p = min(p, max_tokens)
    return " ".join(words[:p]).strip()

def fixed_wrong_pool_for_row(row: pd.Series, tokenizer=None) -> List[str]:
    q, a, cat = row["question"], row["answer"], row["category"]
    pool = []
    # external weak model traces
    for wp in parse_external_wrong_prefixes(row.get("external_wrong_prefixes", None)):
        pred = extract_boxed(wp)
        if pred and answers_match(pred, a):
            continue
        pool.append(truncate_prefix_by_ratio(wp, cfg.prefix_ratio_rho, tokenizer, max_tokens=cfg.response_budget_tokens//3))
    # deterministic structured corruptions
    wrongs = category_wrong_answers(q, a, cat)
    for i, wa in enumerate(wrongs):
        pool.append(truncate_prefix_by_ratio(build_wrong_prefix(q, a, wa, cat, i), cfg.prefix_ratio_rho, tokenizer, max_tokens=cfg.response_budget_tokens//3))
    # Dedup; enforce parseable wrongness when boxed exists.
    clean = []
    for p in pool:
        p = re.sub(r"\s+", " ", str(p)).strip()
        if not p or p in clean:
            continue
        pred = extract_boxed(p)
        if pred and answers_match(pred, a):
            continue
        clean.append(p)
    return clean[:cfg.max_wrong_prefixes_per_row]

# Build prefix pools without tokenizer first; after tokenizer load we rebuild token-exact if desired.
train_df["wrong_prefix_pool"] = train_df.apply(lambda r: fixed_wrong_pool_for_row(r, tokenizer=None), axis=1)
train_df["wrong_prefix_count"] = train_df.wrong_prefix_pool.map(len)
print(train_df[["category", "wrong_prefix_count"]].groupby("category").agg(["count", "mean", "min", "max"]))
assert train_df.wrong_prefix_count.sum() > 0, "No wrong prefixes were generated from real rows. Attach weak rollouts or check answer columns."

In [ ]:
# ============================================================
# 7. Recovery targets and DenoiseRL-style record builder
# ============================================================

RECOVERY_INSTRUCTION = (
    "You are recovering from a noisy prefix. The prefix may contain wrong assumptions, arithmetic, "
    "or a wrong boxed answer. Re-anchor to the original problem, keep only useful facts, repair the error, "
    "and put the final answer in \\boxed{} exactly."
)

MAIN_INSTRUCTION = "Solve the problem. Put the final answer in \\boxed{} exactly."

CATEGORY_REPAIR_HINTS = {
    "cipher": "Rebuild the mapping from constraints; do not trust the prefix's decoded text unless verified.",
    "cryptarithm": "Check digit uniqueness, leading-zero rules, operation consistency, and only then commit.",
    "bit_manipulation": "Test candidate bit operations against all examples before choosing the output.",
    "numeral_system": "Track base conversions explicitly and verify by converting back.",
    "unit_conversion": "Keep dimensional units attached and verify the scale factor.",
    "physics_gravity": "Use the physical equation with units; reject dimensionally invalid prefix steps.",
    "equation_numeric": "Substitute back or verify numerically before boxing.",
    "logic": "Separate assumptions from conclusions and test contradictions.",
    "general": "Recompute from the original problem statement and verify the boxed answer."
}

def build_main_output(row: pd.Series) -> str:
    cat = row["category"]
    hint = CATEGORY_REPAIR_HINTS.get(cat, CATEGORY_REPAIR_HINTS["general"])
    # Compact verified target. It avoids fabricating long hidden reasoning while enforcing final-answer discipline.
    return f"I will solve from the original constraints and verify the result. {hint} The verified final answer is {boxed(row['answer'])}."

def build_recovery_output(row: pd.Series, noisy_prefix: str) -> str:
    cat = row["category"]
    hint = CATEGORY_REPAIR_HINTS.get(cat, CATEGORY_REPAIR_HINTS["general"])
    return (
        "Recovery: the prefix is not trusted as a proof. "
        f"{hint} Rechecking against the original question gives the verified answer {boxed(row['answer'])}."
    )

def build_prompt(question: str, instruction: str, noisy_prefix: Optional[str] = None) -> str:
    if noisy_prefix is None:
        return f"System: You are a precise reasoning model.\nUser: {instruction}\n\nProblem:\n{question}\nAssistant:"
    return (
        f"System: You are a precise reasoning model.\n"
        f"User: {instruction}\n\nProblem:\n{question}\n\nNoisy prefix to recover from:\n{noisy_prefix}\nAssistant:"
    )

def build_training_records(df: pd.DataFrame) -> List[Dict[str, Any]]:
    records = []
    # CPCR order: stable/verifiable categories first, then weak categories; this reduces contamination.
    ordered = df.copy()
    ordered["cat_order"] = ordered.category.map(lambda c: CATEGORY_ORDER.index(c) if c in CATEGORY_ORDER else len(CATEGORY_ORDER))
    ordered = ordered.sort_values(["cat_order", "id"]).reset_index(drop=True)

    for _, row in ordered.iterrows():
        cat = row["category"]
        w = CATEGORY_WEIGHTS.get(cat, 1.0)
        # Main records preserve solve-from-scratch behavior.
        main_reps = max(1, int(round(cfg.main_rollouts_n / 12 * w)))
        for j in range(main_reps):
            prompt = build_prompt(row["question"], MAIN_INSTRUCTION)
            out = build_main_output(row)
            records.append({
                "id": row["id"], "category": cat, "kind": "main", "prompt": prompt,
                "noisy_prefix": "", "target": out, "answer": row["answer"], "loss_weight": 1.0,
            })
        # Denoise records teach recovery from a fixed W(q) pool.
        pool = list(row["wrong_prefix_pool"])
        if not pool:
            continue
        denoise_reps = max(1, int(round(cfg.denoise_rollouts_k / 4 * w)))
        # Cycle through pool; do not over-use one corruption style.
        for j in range(min(len(pool), max(cfg.min_wrong_prefixes_per_row, denoise_reps))):
            noisy = pool[j % len(pool)]
            prompt = build_prompt(row["question"], RECOVERY_INSTRUCTION, noisy)
            out = build_recovery_output(row, noisy)
            records.append({
                "id": row["id"], "category": cat, "kind": "denoise", "prompt": prompt,
                "noisy_prefix": noisy, "target": out, "answer": row["answer"], "loss_weight": 1.15,
            })
    if cfg.max_records and len(records) > cfg.max_records:
        # Preserve category mix with seeded sample.
        rng = random.Random(SEED)
        records = rng.sample(records, cfg.max_records)
    return records

records = build_training_records(train_df)
records_df = pd.DataFrame(records)
records_path = Path(cfg.work_root) / "denoiserl_training_records.jsonl"
records_df.to_json(records_path, orient="records", lines=True, force_ascii=False)
print("Training records:", len(records_df), "saved:", records_path)
print(records_df.kind.value_counts().to_string())
print(records_df.category.value_counts().to_string())
print(records_df.head(2).to_string(max_colwidth=180))
assert len(records_df) > 0
assert records_df.target.str.contains(r"\\boxed\{").all(), "Every target must contain boxed answer."

In [ ]:
# ============================================================
# 8. Load tokenizer/model and build masked dataset
# ============================================================

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback
    from transformers import BitsAndBytesConfig, DataCollatorForSeq2Seq
except Exception as e:
    raise RuntimeError("transformers is required. Attach/install compatible transformers.") from e

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
except Exception as e:
    raise RuntimeError("peft is required. Attach/install PEFT.") from e

try:
    import bitsandbytes as bnb
    HAS_BNB = True
except Exception:
    HAS_BNB = False

# Auto-discover local model path if the configured HF id is not available offline.
def discover_model_path(preferred: str) -> str:
    if preferred and Path(preferred).exists():
        return preferred
    # Search Kaggle inputs for adapter/base model configs.
    roots = [Path(cfg.input_root)]
    model_dirs = []
    for root in roots:
        if not root.exists(): continue
        for p in root.rglob("config.json"):
            d = p.parent
            if any((d / name).exists() for name in ["model.safetensors", "pytorch_model.bin", "model-00001-of-00001.safetensors"]):
                model_dirs.append(d)
            elif list(d.glob("model-*.safetensors")):
                model_dirs.append(d)
    # Prefer Nemotron-like names.
    for d in model_dirs:
        if "nemotron" in str(d).lower():
            return str(d)
    if model_dirs:
        return str(model_dirs[0])
    return preferred

model_path = discover_model_path(cfg.base_model_name_or_path)
print("Using model path/id:", model_path)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
tokenizer.padding_side = "right"

# Rebuild token-ratio prefix pools now tokenizer exists.
train_df["wrong_prefix_pool"] = train_df.apply(lambda r: fixed_wrong_pool_for_row(r, tokenizer=tokenizer), axis=1)
records = build_training_records(train_df)
records_df = pd.DataFrame(records)
records_df.to_json(records_path, orient="records", lines=True, force_ascii=False)
print("Rebuilt token-aware records:", len(records_df))

class DenoiseMaskedDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_len: int):
        self.rows = frame.to_dict("records")
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = r["prompt"].rstrip() + " "
        target = str(r["target"]).strip()
        full = prompt + target
        # Tokenize prompt separately to mask all prompt/noisy-prefix tokens.
        p_ids = self.tok(prompt, add_special_tokens=False, truncation=True, max_length=self.max_len).input_ids
        enc = self.tok(full, add_special_tokens=True, truncation=True, max_length=self.max_len)
        input_ids = enc.input_ids
        labels = list(input_ids)
        mask_upto = min(len(p_ids), len(labels))
        labels[:mask_upto] = [-100] * mask_upto
        # If truncation removed all target labels, keep at least last token trainable.
        if all(x == -100 for x in labels) and labels:
            labels[-1] = input_ids[-1]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(enc.attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# Split deterministically.
shuf = records_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
valid_n = max(1, min(len(shuf)//20, 512)) if len(shuf) >= 20 else 0
valid_records = shuf.iloc[:valid_n].reset_index(drop=True) if valid_n else shuf.iloc[:0]
train_records = shuf.iloc[valid_n:].reset_index(drop=True) if valid_n else shuf
train_ds = DenoiseMaskedDataset(train_records, tokenizer, cfg.max_seq_length)
valid_ds = DenoiseMaskedDataset(valid_records, tokenizer, cfg.max_seq_length) if len(valid_records) else None

# Mask red-team test: prompt/noisy prefix must be ignored by loss.
sample_item = train_ds[0]
assert (sample_item["labels"] == -100).sum().item() > 0, "Prompt/prefix labels were not masked."
assert (sample_item["labels"] != -100).sum().item() > 0, "Target continuation has no trainable labels."
print("Dataset ready:", len(train_ds), "train /", len(valid_records), "valid")

In [ ]:
# ============================================================
# 9. PEFT model construction
# ============================================================

def load_base_model_for_lora(model_path: str):
    dtype = torch.bfloat16 if cfg.bf16 and torch.cuda.is_available() else torch.float16 if torch.cuda.is_available() else torch.float32
    quant_config = None
    if cfg.use_4bit and HAS_BNB and torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
    print("Loading base model. 4bit:", bool(quant_config), "dtype:", dtype)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        quantization_config=quant_config,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False
    if cfg.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    if quant_config is not None:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.gradient_checkpointing)
    return model

def infer_lora_targets(model) -> List[str]:
    preferred = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    names = set()
    for name, module in model.named_modules():
        last = name.split(".")[-1]
        if last in preferred:
            names.add(last)
    if names:
        return sorted(names, key=lambda x: preferred.index(x) if x in preferred else 999)
    # Generic backup for nonstandard remote-code module names.
    linear_suffixes = set()
    for name, module in model.named_modules():
        cls = module.__class__.__name__.lower()
        if "linear" in cls and not any(skip in name.lower() for skip in ["lm_head", "embed"]):
            linear_suffixes.add(name.split(".")[-1])
    if not linear_suffixes:
        raise RuntimeError("Could not infer LoRA target modules.")
    return sorted(linear_suffixes)

if cfg.train_adapter:
    model = load_base_model_for_lora(model_path)
    target_modules = infer_lora_targets(model)
    print("LoRA target modules:", target_modules)
    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
        use_rslora=True,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
else:
    model = None
    print("TRAIN_ADAPTER=0: skipping model load/training.")

In [ ]:
# ============================================================
# 10. Optional DenoiseGRPO core math
# ============================================================
# This is a real guarded implementation of the DenoiseRL objective core.
# Default Kaggle adapter flow uses masked DenoiseSFT because 30B online RL rollouts are usually too expensive.

class DenoiseGRPOCore:
    def __init__(self, clip_low=0.2, clip_high=0.2, eps=1e-6):
        self.clip_low = clip_low
        self.clip_high = clip_high
        self.eps = eps

    @staticmethod
    def group_advantages(rewards: torch.Tensor, group_ids: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        adv = torch.zeros_like(rewards, dtype=torch.float32)
        for gid in group_ids.unique():
            idx = (group_ids == gid)
            r = rewards[idx].float()
            mu = r.mean()
            sigma = torch.sqrt(((r - mu) ** 2).mean() + eps)
            adv[idx] = (r - mu) / sigma
        return adv

    @staticmethod
    def token_logprobs(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # logits [B,T,V], labels [B,T]; logprob of labels at each token. Caller handles shift alignment.
        logp = torch.log_softmax(logits, dim=-1)
        safe_labels = labels.clamp_min(0)
        return torch.gather(logp, -1, safe_labels.unsqueeze(-1)).squeeze(-1)

    def clipped_surrogate_loss(self, new_logp, old_logp, advantages, loss_mask):
        # advantages [B], logp/mask [B,T]
        ratio = torch.exp(new_logp - old_logp)
        adv = advantages[:, None]
        unclipped = ratio * adv
        clipped = torch.clamp(ratio, 1.0 - self.clip_low, 1.0 + self.clip_high) * adv
        objective = torch.minimum(unclipped, clipped)
        denom = loss_mask.sum().clamp_min(1.0)
        # negative because optimizers minimize
        return -(objective * loss_mask).sum() / denom

    def build_loss_mask(self, labels: torch.Tensor, prefix_token_counts: torch.Tensor) -> torch.Tensor:
        # Main rollouts have prefix count 0 beyond prompt mask; denoise rollouts mask off-policy prefix too.
        mask = (labels != -100).float()
        for i, p in enumerate(prefix_token_counts.tolist()):
            if p > 0:
                mask[i, :int(p)] = 0.0
        return mask

print("DenoiseGRPOCore loaded. Online GRPO is guarded by RUN_ONLINE_GRPO=1.")

In [ ]:
# ============================================================
# 11. Train with checkpoint/restart safety
# ============================================================

class TimeBudgetCallback(TrainerCallback):
    def __init__(self, hours: float):
        self.seconds = float(hours) * 3600.0
        self.start = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start
        # Leave enough time for adapter save/package.
        if elapsed > max(60.0, self.seconds - 600.0):
            print(f"Time budget reached at step {state.global_step}; requesting stop.")
            control.should_training_stop = True
        return control

class AdapterCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, model=None, **kwargs):
        if model is not None and hasattr(model, "save_pretrained"):
            step_dir = Path(args.output_dir) / f"adapter_step_{state.global_step}"
            step_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(step_dir, safe_serialization=True)
            print("Saved adapter checkpoint:", step_dir)
        return control

if cfg.train_adapter:
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)
    args = TrainingArguments(
        output_dir=cfg.output_dir,
        overwrite_output_dir=True,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        num_train_epochs=cfg.num_train_epochs,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        logging_steps=cfg.logging_steps,
        save_steps=cfg.save_steps,
        save_strategy="steps",
        eval_strategy="steps" if valid_ds is not None and len(valid_ds) else "no",
        eval_steps=cfg.save_steps if valid_ds is not None and len(valid_ds) else None,
        bf16=cfg.bf16 and torch.cuda.is_available(),
        fp16=(not cfg.bf16) and torch.cuda.is_available(),
        report_to="none",
        remove_unused_columns=False,
        optim="paged_adamw_8bit" if HAS_BNB and torch.cuda.is_available() else "adamw_torch",
        gradient_checkpointing=cfg.gradient_checkpointing,
        max_grad_norm=1.0,
        dataloader_num_workers=0,
        save_total_limit=3,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds if valid_ds is not None and len(valid_ds) else None,
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[TimeBudgetCallback(cfg.train_time_budget_hours), AdapterCheckpointCallback()],
    )
    print("Starting DenoiseRL masked-continuation LoRA training...")
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    # Explicit adapter save; this is the artifact Kaggle needs.
    model.save_pretrained(cfg.output_dir, safe_serialization=True)
    print("Final adapter saved:", cfg.output_dir)
else:
    print("Training skipped by config.")

In [ ]:
# ============================================================
# 12. Local generation eval, optional
# ============================================================

@torch.no_grad()
def generate_answer(model, tokenizer, question: str, max_new_tokens: int = 256) -> str:
    model.eval()
    prompt = build_prompt(question, MAIN_INSTRUCTION)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg.max_seq_length).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

def local_eval_sample(model, tokenizer, df: pd.DataFrame, n: int = 64) -> pd.DataFrame:
    sample = df.sample(min(n, len(df)), random_state=SEED).reset_index(drop=True)
    rows = []
    for _, r in sample.iterrows():
        pred_text = generate_answer(model, tokenizer, r.question)
        pred = extract_boxed(pred_text)
        ok = answers_match(pred, r.answer)
        rows.append({"id": r.id, "category": r.category, "gold": r.answer, "pred": pred, "ok": ok, "text": pred_text[:500]})
    return pd.DataFrame(rows)

if cfg.run_local_generation_eval and cfg.train_adapter and model is not None:
    eval_df = local_eval_sample(model, tokenizer, train_df, n=64)
    eval_path = Path(cfg.work_root) / "local_generation_eval.csv"
    eval_df.to_csv(eval_path, index=False)
    print("Eval accuracy:", eval_df.ok.mean())
    print(eval_df.groupby("category").ok.mean().sort_values(ascending=False).to_string())
    print("Saved eval:", eval_path)
else:
    print("Local generation eval skipped. Set RUN_EVAL=1 to enable.")

In [ ]:
# ============================================================
# 13. Adapter validation and Kaggle package
# ============================================================

def validate_adapter_dir(adapter_dir: str | Path) -> Dict[str, Any]:
    d = Path(adapter_dir)
    if not d.exists():
        raise FileNotFoundError(d)
    cfg_path = d / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Missing adapter_config.json in {d}")
    acfg = json.loads(cfg_path.read_text())
    rank = int(acfg.get("r", acfg.get("rank", -1)))
    if cfg.enforce_rank_limit and rank > cfg.max_lora_rank:
        raise ValueError(f"Adapter rank {rank} exceeds max {cfg.max_lora_rank}")
    weight_files = list(d.glob("adapter_model*.safetensors")) + list(d.glob("adapter_model*.bin"))
    if not weight_files:
        raise FileNotFoundError("Missing adapter_model weights.")
    # Prevent accidental full base model submission.
    forbidden = [p for p in d.glob("*.safetensors") if p.name.startswith("model")]
    if forbidden:
        raise ValueError(f"Full base model weights found in adapter dir: {forbidden[:3]}")
    total_bytes = sum(p.stat().st_size for p in d.rglob("*") if p.is_file())
    return {"adapter_dir": str(d), "rank": rank, "weight_files": [str(p.name) for p in weight_files], "total_mb": total_bytes / 1e6, "peft_type": acfg.get("peft_type")}

def zip_adapter(adapter_dir: str | Path, zip_path: str | Path) -> Path:
    adapter_dir = Path(adapter_dir)
    zip_path = Path(zip_path)
    if zip_path.exists(): zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in adapter_dir.rglob("*"):
            if not p.is_file(): continue
            # Include only adapter/tokenizer/config metadata, not optimizer state or full checkpoints.
            rel = p.relative_to(adapter_dir)
            if any(part.startswith("checkpoint-") for part in rel.parts):
                continue
            if p.name in {"optimizer.pt", "scheduler.pt", "trainer_state.json", "rng_state.pth"}:
                continue
            if p.name.startswith("model-") or p.name == "model.safetensors":
                continue
            zf.write(p, arcname=str(rel))
    return zip_path

if cfg.train_adapter:
    report = validate_adapter_dir(cfg.output_dir)
    print("Adapter validation:", json.dumps(report, indent=2))
    zpath = zip_adapter(cfg.output_dir, cfg.package_zip)
    print("Adapter package:", zpath, "size MB:", zpath.stat().st_size/1e6)
    # Re-open zip to ensure adapter_config is at root.
    with zipfile.ZipFile(zpath) as zf:
        names = set(zf.namelist())
        assert "adapter_config.json" in names, "adapter_config.json must be at zip root."
        assert any(n.startswith("adapter_model") for n in names), "adapter model weights must be at zip root."
else:
    print("Adapter validation skipped because training skipped.")

In [ ]:
# ============================================================
# 14. Final red-team report
# ============================================================

redteam = {
    "real_train_path": str(train_path),
    "train_rows": int(len(train_df)),
    "training_records": int(len(records_df)),
    "record_kind_counts": records_df.kind.value_counts().to_dict(),
    "category_counts": train_df.category.value_counts().to_dict(),
    "wrong_prefix_total": int(train_df.wrong_prefix_count.sum()),
    "prefix_ratio_rho": cfg.prefix_ratio_rho,
    "main_rollouts_n": cfg.main_rollouts_n,
    "denoise_rollouts_k": cfg.denoise_rollouts_k,
    "lora_rank": cfg.lora_r,
    "output_dir": cfg.output_dir,
    "package_zip": cfg.package_zip,
}

# Core invariants.
assert redteam["train_rows"] > 0
assert redteam["wrong_prefix_total"] > 0
assert cfg.lora_r <= cfg.max_lora_rank
assert records_df.target.map(lambda x: "\\boxed{" in str(x)).all()
assert not records_df.prompt.str.contains("lorem ipsum|dummy dataset|mock dataset|toy dataset", case=False, regex=True).any()

report_path = Path(cfg.work_root) / "denoiserl_redteam_report.json"
report_path.write_text(json.dumps(redteam, indent=2), encoding="utf-8")
print(json.dumps(redteam, indent=2))
print("Red-team report saved:", report_path)
print("DONE")